In [1]:
#manupulação de dados em tabelas
import pandas as pd 
#plots de gráficos
import matplotlib.pyplot as plt
#manupalação de vetores
import numpy as np 

#biblioteca spacy
import spacy
#OBS: Caso não tenha o pacote em português, execute no terminal o comando: python3 -m spacy download pt

# biblioteca string - Nativa do python
import string 

#Os stop words são oriundo da biblioteca spacy
from spacy.lang.pt.stop_words import STOP_WORDS

In [2]:
pln=spacy.load("pt_core_news_sm")
stop_words=STOP_WORDS
pontuacoes=string.punctuation
pontuacoes=pontuacoes+"..."+' '

# remove da lista de stop words alguns elementos importantes
stop_words.remove('bom')
stop_words.remove('muito')
stop_words.remove('não')
stop_words.remove('nem')

In [3]:
def processamento(texto):
    # texto em minuscula
    texto=texto.lower()
    documento=pln(texto)
    
    #removendo stop words
    lista_tokens_1=[]
    for p in documento:
        if (p.text in stop_words)==False:
            lista_tokens_1.append(p)
    #removendo pontuações      
    lista_tokens_2=[]
    for p in lista_tokens_1:
        if (p.text in pontuacoes)==False:
            lista_tokens_2.append(p)
    #lematização de tokens        
    lista_tokens_3=[]
    for p in lista_tokens_2:
        lista_tokens_3.append(p.lemma_)

    return lista_tokens_3

In [4]:
df=pd.read_csv('olist_order_reviews_dataset.csv')
# Amostrado da tabela
df.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [5]:
df['review_creation_date']=pd.to_datetime(df['review_creation_date'])
# encontrando o ano de publicação
ano=[]
for i in range(len(df)):
    ano.append(df['review_creation_date'].iloc[i].year)
df['ANO']=ano

# Seleção de comentários de 2018
df=df[df['ANO']==2018].reset_index(drop=True)

#remover linhas duplicadas
df.drop_duplicates(subset='review_id',inplace=True)
# selecionar apenas algumas colunas releantes
df=df[['review_comment_title','review_comment_message','review_score']].reset_index(drop=True)
# preencher campos vazios
df.fillna('',inplace=True)
# reestruturação dos comentários
df['review']=df['review_comment_title']+ ' '+df['review_comment_message']
# remoção de comentário vazios
df['review']=df['review'].replace(' ',np.nan)
df=df.dropna(subset="review").reset_index(drop=True)

# utilizar uma amostra do dado 
df=df.sample(1000).reset_index(drop=True)

In [6]:
df.review_score.value_counts()

review_score
5    485
1    215
4    157
3     91
2     52
Name: count, dtype: int64

# Vetorizar textos

In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import seaborn as sns 

In [8]:
corpus=list(df['review'].values)

In [9]:
# essa etapa demora um pouco
vectorizer = CountVectorizer(tokenizer=processamento,max_features=300,stop_words=None,token_pattern=None)
vectorizer.fit(corpus)

CountVectorizer(max_features=300, token_pattern=None,
                tokenizer=<function processamento at 0x7f2d31810f70>)

In [10]:
vocabulario=vectorizer.get_feature_names()

/home/jlbdearaujo/.local/lib/python3.8/site-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


In [11]:
X_bow = vectorizer.transform(corpus)

In [12]:
from sklearn.decomposition import LatentDirichletAllocation, NMF
import pyLDAvis
import pyLDAvis.sklearn as sklearn_lda

# LDA

In [13]:
lda = LatentDirichletAllocation(n_components=8, random_state=42)
lda.fit(X_bow)

/home/jlbdearaujo/.local/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


LatentDirichletAllocation(n_components=8, random_state=42)

In [ ]:
vis_lda = sklearn_lda.prepare(lda, X_bow, vectorizer, mds='tsne')
pyLDAvis.save_html(vis_lda, 'pyldavis_LDA.html')